# 08 Rdds - Annotated Notebook

This notebook includes step-by-step markdown sections and English code comments.

## Step 1 - Import required libraries

In [1]:
# Import required libraries.
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

22/02/21 22:25:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


## Step 2 - Load data into a Spark DataFrame

In [2]:
# Load data into a Spark DataFrame.
df_green = spark.read.parquet('data/pq/green/*/*')

```
SELECT 
    date_trunc('hour', lpep_pickup_datetime) AS hour, 
    PULocationID AS zone,

    SUM(total_amount) AS amount,
    COUNT(1) AS number_records
FROM
    green
WHERE
    lpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
```

## Step 3 - Work with RDD transformations

In [8]:
# Work with RDD transformations.
rdd = df_green \
    .select('lpep_pickup_datetime', 'PULocationID', 'total_amount') \
    .rdd

## Step 4 - Import required libraries

In [13]:
# Import required libraries.
from datetime import datetime

## Step 5 - Run the next processing step

In [19]:
# Run the next processing step.
start = datetime(year=2020, month=1, day=1)

def filter_outliers(row):
    return row.lpep_pickup_datetime >= start

## Step 6 - Work with RDD transformations

In [21]:
# Work with RDD transformations.
rows = rdd.take(10)
row = rows[0]

## Step 7 - Run the next processing step

In [29]:
# Run the next processing step.
row

Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 16, 19, 49, 27), PULocationID=260, total_amount=14.3)

## Step 8 - Define a helper function

In [31]:
# Define a helper function.
def prepare_for_grouping(row): 
    hour = row.lpep_pickup_datetime.replace(minute=0, second=0, microsecond=0)
    zone = row.PULocationID
    key = (hour, zone)
    
    amount = row.total_amount
    count = 1
    value = (amount, count)

    return (key, value)

## Step 9 - Define a helper function

In [34]:
# Define a helper function.
def calculate_revenue(left_value, right_value):
    left_amount, left_count = left_value
    right_amount, right_count = right_value
    
    output_amount = left_amount + right_amount
    output_count = left_count + right_count
    
    return (output_amount, output_count)

## Step 10 - Import required libraries

In [39]:
# Import required libraries.
from collections import namedtuple

## Step 11 - Run the next processing step

In [40]:
# Run the next processing step.
RevenueRow = namedtuple('RevenueRow', ['hour', 'zone', 'revenue', 'count'])

## Step 12 - Define a helper function

In [41]:
# Define a helper function.
def unwrap(row):
    return RevenueRow(
        hour=row[0][0], 
        zone=row[0][1],
        revenue=row[1][0],
        count=row[1][1]
    )

## Step 13 - Import required libraries

In [45]:
# Import required libraries.
from pyspark.sql import types

## Step 14 - Run the next processing step

In [46]:
# Run the next processing step.
result_schema = types.StructType([
    types.StructField('hour', types.TimestampType(), True),
    types.StructField('zone', types.IntegerType(), True),
    types.StructField('revenue', types.DoubleType(), True),
    types.StructField('count', types.IntegerType(), True)
])

## Step 15 - Work with RDD transformations

In [47]:
# Work with RDD transformations.
df_result = rdd \
    .filter(filter_outliers) \
    .map(prepare_for_grouping) \
    .reduceByKey(calculate_revenue) \
    .map(unwrap) \
    .toDF(result_schema) 

## Step 16 - Write results to storage

In [50]:
# Write results to storage.
df_result.write.parquet('tmp/green-revenue')

## Step 17 - Run the next processing step

In [55]:
# Run the next processing step.
columns = ['VendorID', 'lpep_pickup_datetime', 'PULocationID', 'DOLocationID', 'trip_distance']

duration_rdd = df_green \
    .select(columns) \
    .rdd

## Step 18 - Import required libraries

In [67]:
# Import required libraries.
import pandas as pd

## Step 19 - Work with RDD transformations

In [68]:
# Work with RDD transformations.
rows = duration_rdd.take(10)

## Step 20 - Run the next processing step

In [81]:
# Run the next processing step.
df = pd.DataFrame(rows, columns=columns)

## Step 21 - Run the next processing step

In [74]:
# Run the next processing step.
columns

['VendorID',
 'lpep_pickup_datetime',
 'PULocationID',
 'DOLocationID',
 'trip_distance']

## Step 22 - Define a helper function

In [76]:
#model = ...

def model_predict(df):
#     y_pred = model.predict(df)
    y_pred = df.trip_distance * 5
    return y_pred

## Step 23 - Define a helper function

In [98]:
# Define a helper function.
def apply_model_in_batch(rows):
    df = pd.DataFrame(rows, columns=columns)
    predictions = model_predict(df)
    df['predicted_duration'] = predictions

    for row in df.itertuples():
        yield row

## Step 24 - Work with RDD transformations

In [102]:
# Work with RDD transformations.
df_predicts = duration_rdd \
    .mapPartitions(apply_model_in_batch)\
    .toDF() \
    .drop('Index')

## Step 25 - Inspect DataFrame content and size

In [104]:
# Inspect DataFrame content and size.
df_predicts.select('predicted_duration').show()

+------------------+
|predicted_duration|
+------------------+
|             12.95|
|             31.25|
|              14.0|
|             12.75|
|               0.1|
|             11.05|
|11.299999999999999|
|54.349999999999994|
|             15.25|
|             91.75|
|             12.25|
|               3.1|
|               7.5|
|11.899999999999999|
| 78.89999999999999|
|              4.45|
|              23.2|
|              4.85|
|              6.65|
|              15.1|
+------------------+
only showing top 20 rows



## Step 26 - Run this processing step

In [ ]:
# Run this processing step.
